In [18]:
# pure FFT / IFFT spectral subtraction implementation, no noisereduce,
# no ML, no fancy DSP libraries.
# This is classic textbook spectral noise reduction,
# and it’s exactly what you’d later port to C / embedded / ESP32.

#What this code does
#	1.	Load WAV audio
#	2.	Add white noise
#	3.	Estimate noise spectrum (FFT only)
#	4.	Perform spectral subtraction frame-by-frame
#	5.	Reconstruct signal with IFFT + overlap-add
#	6.	Play:
#	•	original
#	•	noisy
#	•	denoised


# Required packages
# pip install numpy scipy soundfile sounddevice

import numpy as np
import soundfile as sf
from IPython.display import Audio, display
import requests
from scipy.io import wavfile
import io
import requests
import librosa

In [19]:
# 1. Define the RAW GitHub URL   RAW!!!!!!!!!!  RAW!!!!!!
# The original URL pointed to the HTML preview; this one points to the raw data.
url = 'https://raw.githubusercontent.com/rfurch/unlpam_noise/refs/heads/main/data/originalAudioFile.mp3'

file_name = "originalAudioFile.mp3"

r = requests.get(url)
with open(file_name, "wb") as f:
    f.write(r.content)

print("Downloaded:", file_name)

Downloaded: originalAudioFile.mp3


In [20]:

y, sr = librosa.load("originalAudioFile.mp3", sr=None)

# Display the audio player (optional)
print(f"Waveform shape: {y.shape}")
print(f"Sampling rate: {sr} Hz")

# amplification
y = y * 10

Waveform shape: (375552,)
Sampling rate: 44100 Hz


In [21]:
Audio(y, rate=sr, autoplay=True, normalize=False)


In [22]:
# ----------------------------
# Add white noise
# ----------------------------
noise_level = 0.02
y_noisy = y + noise_level * np.random.randn(len(y))

In [23]:
Audio(y_noisy, rate=sr, autoplay=True, normalize=False)


In [24]:
# ----------------------------
# FFT parameters
# ----------------------------
frame_size = 1024
hop_size = frame_size // 2
window = np.hanning(frame_size)

# ----------------------------
# Estimate noise spectrum
# Use first 0.5 sec as noise
# ----------------------------
noise_duration = 0.9
noise_samples = int(noise_duration * sr)
num_noise_frames = (noise_samples - frame_size) // hop_size

noise_mag = np.zeros(frame_size)

for i in range(num_noise_frames):
    start = i * hop_size
    frame = y_noisy[start:start + frame_size] * window
    spectrum = np.fft.fft(frame)
    noise_mag += np.abs(spectrum)

noise_mag /= num_noise_frames


In [25]:
# ----------------------------
# Spectral subtraction
# ----------------------------
output = np.zeros(len(y_noisy))
window_sum = np.zeros(len(y_noisy))

for i in range(0, len(y_noisy) - frame_size, hop_size):
    frame = y_noisy[i:i + frame_size] * window
    spectrum = np.fft.fft(frame)

    mag = np.abs(spectrum)
    phase = np.angle(spectrum)

    ######################################################
    # Subtract noise magnitude  (alternative A)
    #clean_mag = mag - noise_mag

    # Subtract noise magnitude  (alternative B)
    alpha = 1.5   # 1.2–2.0 típico
    clean_mag = mag - alpha * noise_mag
    ######################################################

    # spectral floor, alternative A
    #clean_mag = np.maximum(clean_mag, 0.0)

    # spectral floor, alternative B
    beta = 0.02   # 1–5% del ruido
    floor = beta * noise_mag
    clean_mag = np.maximum(clean_mag, floor)


    # Reconstruct complex spectrum
    clean_spectrum = clean_mag * np.exp(1j * phase)

    # IFFT
    clean_frame = np.fft.ifft(clean_spectrum).real

    # Overlap-add
    output[i:i + frame_size] += clean_frame * window
    window_sum[i:i + frame_size] += window**2


# Normalize overlap-add
#nonzero = window_sum > 1e-8
#output[nonzero] /= window_sum[nonzero]


In [26]:
Audio(output, rate=sr, autoplay=True)
